# 9.3 批处理优化 (Batch Processing)

批处理优化通过高效组织推理请求来提升GPU利用率和吞吐量。

本节涵盖：
- Continuous Batching
- 动态批处理

## 1. Continuous Batching

**目的**：消除请求间的空闲等待

**基本原理**：在iteration级别调度请求，当一个请求生成完成后立即从batch中移除并加入新请求，而不是等待整个batch完成。

**Static Batching vs Continuous Batching**：
- Static：等所有请求完成才能处理新请求，GPU空闲
- Continuous：每个iteration都可以加入/移除请求，GPU始终忙碌

**吞吐量提升**：通常2-4倍

In [ ]:
import torch
import random

random.seed(42)

n_requests = 8
output_lengths = [random.randint(5, 20) for _ in range(n_requests)]
max_len = max(output_lengths)

print('=== Continuous Batching ===')
print(f'Requests: {n_requests}')
print(f'Output lengths: {output_lengths}')
print(f'Max length: {max_len}')

static_steps = max_len
static_gpus = sum(output_lengths) + (max_len * n_requests - sum(output_lengths))

continuous_steps = 0
active = list(range(n_requests))
remaining = output_lengths.copy()
total_compute = 0
while active:
    continuous_steps += 1
    total_compute += len(active)
    done = []
    for i in active:
        remaining[i] -= 1
        if remaining[i] == 0:
            done.append(i)
    for d in done:
        active.remove(d)

static_total = max_len * n_requests
static_waste = static_total - sum(output_lengths)
print(f'\nStatic batching:')
print(f'  Total GPU steps: {static_total} ({max_len} steps x {n_requests} requests)')
print(f'  Wasted steps: {static_waste} ({static_waste/static_total:.1%})')
print(f'\nContinuous batching:')
print(f'  Total GPU steps: {total_compute}')
print(f'  Wasted steps: 0 (requests leave immediately when done)')
print(f'  Efficiency gain: {static_total/total_compute:.1f}x')
print(f'\nKey: Continuous batching eliminates padding waste.')
print(f'Used by vLLM, TGI, TensorRT-LLM for production serving.')

## 2. 动态批处理

**目的**：自动将并发请求组合成批次

**基本原理**：收集一定时间窗口内的请求，将它们打包成一个batch一起处理，在延迟和吞吐量之间取得平衡。

**关键参数**：
- 最大batch size：限制同时处理的请求数
- 等待超时：等待更多请求加入batch的最大时间
- 优先级调度：高优先级请求优先处理

In [ ]:
import random

random.seed(42)

max_batch = 4
timeout_ms = 50
requests = [(i, random.randint(10, 100)) for i in range(12)]

print('=== Dynamic Batching ===')
print(f'Max batch size: {max_batch}')
print(f'Timeout: {timeout_ms}ms')
print(f'\nIncoming requests (id, input_tokens):')
for rid, tokens in requests:
    print(f'  Request {rid}: {tokens} tokens')

batches = []
current_batch = []
for rid, tokens in requests:
    current_batch.append((rid, tokens))
    if len(current_batch) >= max_batch:
        batches.append(current_batch)
        current_batch = []
if current_batch:
    batches.append(current_batch)

print(f'\nFormed batches:')
for i, batch in enumerate(batches):
    ids = [r[0] for r in batch]
    max_tokens = max(r[1] for r in batch)
    total_padded = max_tokens * len(batch)
    actual = sum(r[1] for r in batch)
    print(f'  Batch {i}: requests={ids}, max_tokens={max_tokens}, utilization={actual/total_padded:.1%}')

print(f'\nKey: Dynamic batching trades slight latency for higher throughput.')
print(f'Larger batches = better GPU utilization but higher per-request latency.')